# XGBoost — Walmart Store Sales Forecasting

XGBoost ექსპერიმენტების notebook.

**საუკეთესო შედეგი:** target encoding + lag_52 + lag_104 + dept_share + holiday_countdown + extra_holidays, ტუნინგით (lr=0.03, n=1000) — validation WMAE ≈ 1409.

("holiday_countdown", HolidayCountdownAdder()),
    ("extra_holidays", ExtraHolidayAdder()),
    ("preprocessing", WalmartPreprocessor()),

In [47]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


## MLflow / DagsHub ინიციალიზაცია და GitHub-დან preprocessing-ის იმპორტი

In [48]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

!wget -q -O preprocessing.py "https://raw.githubusercontent.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/main/preprocessing.py"
from preprocessing import run_pipeline, weighted_mae, THANKSGIVING, CHRISTMAS

In [49]:
%pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='izere23',
             repo_name='ML-Final-Walmart-Recruiting-Store-Sales-Forecasting',
             mlflow=True)

import mlflow
import mlflow.sklearn

Note: you may need to restart the kernel to use updated packages.


Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

## მონაცემების ჩატვირთვა და train/validation split

ვიყენებტ `run_pipeline`-ს: `merge` → `grid-fill` → `markdown flags` → `calendar features`. train/validation ხდება time split-ით

In [50]:
train_part, valid_part, train_full, test_full = run_pipeline(
    data_dir="/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting",
    out_dir="/kaggle/working/processed",
)

for df in (train_part, valid_part, test_full):
    if "IsHoliday" not in df.columns and "IsHoliday_x" in df.columns:
        df.rename(columns={"IsHoliday_x": "IsHoliday"}, inplace=True)
        df.drop(columns=["IsHoliday_y"], inplace=True)

TARGET = "Weekly_Sales"
train = train_part.dropna(subset=[TARGET]).copy()
valid = valid_part.dropna(subset=[TARGET]).copy()

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val,   y_val   = valid.drop(columns=[TARGET]), valid[TARGET]

print(X_train.shape, X_val.shape)

Expected rows if no gaps: 428409
Actual rows: 380107
Missing (Store,Dept,Date) combos filled in: 48302
Expected rows if no gaps: 476333
Actual rows: 421570
Missing (Store,Dept,Date) combos filled in: 54763
Saved parquet files to /kaggle/working/processed
(380107, 32) (41463, 31)


## Preprocessing და Feature Engineering transformer-ები

### WalmartPreprocessor — საბაზისო გასუფთავება

ეს transformer მოდელამდე ასუფთავებს DataFrame-ს ორ ნაბიჯად:
- **შლის leak სვეტებს** — `Weekly_Sales_clipped`, `is_negative_sales` (ორივე target-იდან არის ნაწარმოები, ანუ პასუხს ამჟღავნებდა) და `Date` (datetime, რომელსაც XGBoost ვერ იღებს — მისი ინფორმაცია უკვე `Year/Month/WeekOfYear`-შია), ასევე `was_grid_filled`.
- **ასწორებს ტიპებს** — `Store`, `Dept`, `Type` გადაჰყავს `category`-ში (რომ ხე მათ იდენტობად და არა რიცხვად დაასპლიტოს), `IsHoliday` — int-ში.

`fit` არაფერს სწავლობს (ფიქსირებული წესებია), ყველა სამუშაო `transform`-შია. `if c in X.columns` დაცვა საშუალებას აძლევს ერთსა და იმავე transformer-ს იმუშაოს train-ზეც და test-ზეც.

In [60]:
class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    """Drop target-leak columns + Date, cast identity columns to category."""
    def __init__(self):
        self.drop_cols = ["Weekly_Sales_clipped", "is_negative_sales", "Date", "was_grid_filled"]
        self.cat_cols = ["Store", "Dept", "Type"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[c for c in self.drop_cols if c in X.columns])
        for c in self.cat_cols:
            if c in X.columns:
                X[c] = X[c].astype("category")
        if "IsHoliday" in X.columns:
            X["IsHoliday"] = X["IsHoliday"].astype(int)
        return X

### StoreDeptTargetEncoder — Target Encoding

ცვლის თითოეულ `(Store, Dept)` წყვილს მისი **ისტორიული საშუალო გაყიდვებით**. ეს პირდაპირ აძლევს ხეს გაყიდვების მასშტაბს, ისე რომ მან 3,331 სერიის იდენტობა ბევრი split-ით აღარ უნდა აღადგინოს.

Smoothing (`m=10`) პატარა ჯგუფებს გლობალური საშუალოსკენ წევს, რომ ხმაური არ შემოვიდეს. **Leakage-safe**: `fit` მხოლოდ train-ზე ითვლის საშუალოებს; val/test იღებს train-ის მნიშვნელობებს და საკუთარ target-ს ვერ ხედავს.

ეს იყო ერთ-ერთი ყველაზე დიდი გაუმჯობესება მთელ პროექტში (feature importance-ში ~65%).

In [61]:
class StoreDeptTargetEncoder(BaseEstimator, TransformerMixin):
    """Add mean historical sales per (Store, Dept), learned on TRAIN only.
       Smoothed toward the global mean so small groups don't get noisy values."""
    def __init__(self, m=10):
        self.m = m  # smoothing strength

    def fit(self, X, y):
        df = X[["Store", "Dept"]].copy()
        df["y"] = np.asarray(y)
        self.global_mean_ = df["y"].mean()
        stats = df.groupby(["Store", "Dept"])["y"].agg(["mean", "count"])
        smoothed = (stats["count"] * stats["mean"] + self.m * self.global_mean_) \
                   / (stats["count"] + self.m)
        self.mapping_ = smoothed.to_dict()      # {(Store, Dept): value}
        return self

    def transform(self, X):
        X = X.copy()
        keys = list(zip(X["Store"], X["Dept"]))
        X["store_dept_te"] = [self.mapping_.get(k, self.global_mean_) for k in keys]
        return X

### Lag52Adder — გასული წლის იმავე კვირის გაყიდვა

ამატებს `lag_52`-ს — იმავე `(Store, Dept)`-ის გაყიდვას **52 კვირის წინ** (გასული წლის იმავე კვირა), მოძებნილს კალენდარული თარიღით. იჭერს წლიურ სეზონურობას (გასული Christmas → ამ Christmas-ს).

**Leakage-safe**: history მხოლოდ train-ისგან შენდება. კრიტიკულია, რომ ეს lag test-ზეც არსებობს — გასული წელი ყოველთვის history-შია, განსხვავებით მოკლე lag-ებისგან (`lag_1` და ა.შ.), რომლებიც future block-ზე ვერ ითვლება.

In [62]:
class Lag52Adder(BaseEstimator, TransformerMixin):
    """Same-week-last-year sales (52 weeks back), looked up by calendar date.
       History built from TRAIN only (fit), so val/test never see their own targets."""
    def fit(self, X, y):
        hist = X[["Store", "Dept", "Date"]].copy()
        hist["sales"] = np.asarray(y)
        hist = hist.drop_duplicates(subset=["Store", "Dept", "Date"])
        self.history_ = hist.set_index(["Store", "Dept", "Date"])["sales"]
        return self

    def transform(self, X):
        X = X.copy()
        lag_date = X["Date"] - pd.Timedelta(weeks=52)
        idx = pd.MultiIndex.from_arrays([X["Store"], X["Dept"], lag_date])
        X["lag_52"] = self.history_.reindex(idx).values
        return X

### გამოუყენებელი transformer-ები (ტესტირებული, უარყოფილი)

`RollingMeanAdder` (trailing 4/8-კვირიანი საშუალო) და `LogTargetRegressor` (log1p target). ორივე ტესტირებულია, მაგრამ საბოლოო მოდელში **არ შედის**:
- **Rolling mean** — short-window features future validation block-ზე ვერ გადადის (მიმდინარე თარიღზე history არ არსებობს), რამაც val WMAE ააფეთქა.
- **Log-target** — WMAE ვერ გააუმჯობესა და fit გააუარესა (log-სივრცე relative error-ს ამცირებს, WMAE კი absolute dollars-ს ითვლის).

In [63]:
class RollingMeanAdder(BaseEstimator, TransformerMixin):
    """Trailing 4/8-week mean sales per (Store,Dept). Uses the most recent
       available history at-or-before each row's date (works on val/test)."""
    def __init__(self, windows=(4, 8)):
        self.windows = windows

    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        h = (h.drop_duplicates(["Store", "Dept", "Date"])
               .sort_values(["Store", "Dept", "Date"]))
        for w in self.windows:
            h[f"roll_{w}"] = (
                h.groupby(["Store", "Dept"])["sales"]
                 .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
            )
        self.history_ = h[["Store", "Dept", "Date"] + [f"roll_{w}" for w in self.windows]]
        return self

    def transform(self, X):
        X = X.copy()
        X["_orig_order"] = np.arange(len(X))
        cols = [f"roll_{w}" for w in self.windows]

        left = X.sort_values("Date")
        right = self.history_.sort_values("Date")

        merged = pd.merge_asof(
            left, right,
            on="Date", by=["Store", "Dept"],
            direction="backward",          # most recent history at-or-before this date
        )
        merged = merged.sort_values("_orig_order").drop(columns="_orig_order")
        for c in cols:
            X[c] = merged[c].values
        X = X.drop(columns="_orig_order")
        return X

In [64]:
from sklearn.base import RegressorMixin, clone

class LogTargetRegressor(BaseEstimator, RegressorMixin):
    """Fit on log1p(y), predict back in raw units via expm1."""
    def __init__(self, model):
        self.model = model

    def fit(self, X, y, **fit_params):
        self.model_ = clone(self.model)
        y_clipped = np.clip(np.asarray(y), 0, None)   # negatives → 0 before log1p
        self.model_.fit(X, np.log1p(y_clipped), **fit_params)
        return self

    def predict(self, X):
        return np.expm1(self.model_.predict(X))

### HolidayCountdownAdder — დღესასწაულამდე დარჩენილი კვირები

ამატებს `weeks_to_thanksgiving` და `weeks_to_christmas`-ს (შეზღუდული ±8-ით). binary flag-ისგან განსხვავებით იჭერს დღესასწაულის წინა **ramp-ს** (გაყიდვები დღესასწაულამდე იზრდება). validation-ზე flat აღმოჩნდა, რადგან ჩვენს val ფანჯარაში (ივლ.–ოქტ.) დღესასწაული არ არის.

In [65]:
class HolidayCountdownAdder(BaseEstimator, TransformerMixin):
    """Weeks-until-Thanksgiving and weeks-until-Christmas (capped at +/-8)."""
    def __init__(self):
        self.thanksgiving = THANKSGIVING  # from preprocessing.py
        self.christmas = CHRISTMAS

    def fit(self, X, y=None):
        return self

    def _weeks_to_nearest(self, dates, holidays):
        d = dates.values.astype("datetime64[D]")
        h = np.array(holidays.values, dtype="datetime64[D]")
        # signed week-diff to the closest holiday date, clipped to [-8, 8]
        diffs = (h[None, :] - d[:, None]) / np.timedelta64(7, "D")
        nearest = diffs[np.arange(len(d)), np.abs(diffs).argmin(axis=1)]
        return np.clip(nearest, -8, 8)

    def transform(self, X):
        X = X.copy()
        X["weeks_to_thanksgiving"] = self._weeks_to_nearest(X["Date"], self.thanksgiving)
        X["weeks_to_christmas"] = self._weeks_to_nearest(X["Date"], self.christmas)
        return X

### LagAdder — განზოგადებული lag

`Lag52Adder`-ის განზოგადება ნებისმიერ lag-ზე. საბოლოო მოდელში იძახება `lags=(52, 104)`-ით — გასული და გუშინდელი წლის იმავე კვირა. `lag_104` მცირე დამატებით signal-ს იძლევა იმ სერიებზე, რომლებსაც 2 წლის ისტორია აქვთ. ორივე deployable-ია.

In [66]:
class LagAdder(BaseEstimator, TransformerMixin):
    """Add sales lagged by N weeks per (Store,Dept), looked up by calendar date.
       History from TRAIN only."""
    def __init__(self, lags=(52,)):
        self.lags = lags

    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        self.history_ = (h.drop_duplicates(["Store", "Dept", "Date"])
                          .set_index(["Store", "Dept", "Date"])["sales"])
        return self

    def transform(self, X):
        X = X.copy()
        for L in self.lags:
            lag_date = X["Date"] - pd.Timedelta(weeks=L)
            idx = pd.MultiIndex.from_arrays([X["Store"], X["Dept"], lag_date])
            X[f"lag_{L}"] = self.history_.reindex(idx).values
        return X

### StoreContextAdder — dept_share

ამატებს `dept_share`-ს — თითოეული dept-ის **საშუალო წილს** მისი store-ის კვირის ჯამურ გაყიდვებში. აძლევს ხეს dept-ის სტრუქტურულ როლს store-ში (პატარა dept დიდ store-ში ≠ იგივე dept პატარა store-ში).

**Leakage-safe**: `dept_share` სტატიკურია თითო `(Store, Dept)`-ზე და train-ზე ისწავლება. თავდაპირველ ვერსიაში `store_week_total` (მიმდინარე კვირის ჯამი) leak-ს იწვევდა — ამოღებულია.

In [67]:
class StoreContextAdder(BaseEstimator, TransformerMixin):
    """Each dept's average share of its store's weekly sales.
       Static per (Store,Dept), learned on TRAIN only — transfers to val/test."""
    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        store_week = h.groupby(["Store", "Date"])["sales"].sum().rename("stot")
        h = h.merge(store_week, on=["Store", "Date"], how="left")
        h["share"] = h["sales"] / h["stot"].replace(0, np.nan)
        self.dept_share_ = h.groupby(["Store", "Dept"])["share"].mean()
        self.global_share_ = h["share"].mean()
        return self

    def transform(self, X):
        X = X.copy()
        keys = list(zip(X["Store"], X["Dept"]))
        X["dept_share"] = [self.dept_share_.get(k, self.global_share_) for k in keys]
        return X

### ExtraHolidayAdder — დამატებითი US დღესასწაულები

flag-ები competition-ის ოთხის მიღმა: Easter, Valentine's, Mother's Day, July 4th, Halloween, Back-to-School. Easter ყველაზე მნიშვნელოვანია (მოძრავია და competition-ის set-ში არ არის). validation-ზე flat/ოდნავ უარესი — ისევ იმიტომ, რომ val ფანჯარა ამ დღესასწაულებს ვერ ხედავს.

In [68]:
class ExtraHolidayAdder(BaseEstimator, TransformerMixin):
    """Flags for US retail holidays beyond the competition's big four.
       Uses the nearest Friday (data is W-FRI) within a week of each date."""
    # dates per year present in the data (2010-2012 train, 2013 test)
    EASTER      = pd.to_datetime(["2010-04-04","2011-04-24","2012-04-08","2013-03-31"])
    VALENTINES  = pd.to_datetime(["2010-02-14","2011-02-14","2012-02-14","2013-02-14"])
    MOTHERS     = pd.to_datetime(["2010-05-09","2011-05-08","2012-05-13","2013-05-12"])
    JULY4       = pd.to_datetime(["2010-07-04","2011-07-04","2012-07-04","2013-07-04"])
    HALLOWEEN   = pd.to_datetime(["2010-10-31","2011-10-31","2012-10-31","2013-10-31"])
    BACK2SCHOOL = pd.to_datetime(["2010-08-20","2011-08-19","2012-08-17","2013-08-16"])

    def fit(self, X, y=None):
        return self

    def _near(self, dates, holidays, days=7):
        d = dates.values.astype("datetime64[D]")
        h = np.array(holidays.values, dtype="datetime64[D]")
        diff = np.abs(h[None, :] - d[:, None]) / np.timedelta64(1, "D")
        return (diff.min(axis=1) <= days).astype(int)

    def transform(self, X):
        X = X.copy()
        X["is_easter"]      = self._near(X["Date"], self.EASTER)
        X["is_valentines"]  = self._near(X["Date"], self.VALENTINES)
        X["is_mothers"]     = self._near(X["Date"], self.MOTHERS)
        X["is_july4"]       = self._near(X["Date"], self.JULY4)
        X["is_halloween"]   = self._near(X["Date"], self.HALLOWEEN)
        X["is_back2school"] = self._near(X["Date"], self.BACK2SCHOOL)
        return X

## საბოლოო მოდელი — All Features + Hand-Tuned

feature set: target encoding + lag_52 + lag_104 + dept_share + holiday countdown + extra holidays. Hand-tuning-მა (`learning_rate=0.03`, `n_estimators=1000`) მისცა საუკეთესო **validation WMAE ≈ 1409**. RandomizedSearchCV-ის tuning-მა ვერ აჯობა ამ hand-tuned კონფიგურაციას.

In [70]:
model = XGBRegressor(
    n_estimators=1000, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", eval_metric="mae",
    tree_method="hist", enable_categorical=True,
    random_state=42, n_jobs=-1,
)

pipeline = Pipeline([
    ("target_encoding", StoreDeptTargetEncoder(m=10)),
    ("lags", LagAdder(lags=(52, 104))),
    ("store_context", StoreContextAdder()),
    ("holiday_countdown", HolidayCountdownAdder()),
    ("extra_holidays", ExtraHolidayAdder()),
    ("preprocessing", WalmartPreprocessor()),
    ("model", model),
])

# MLFlow Log

In [72]:
with mlflow.start_run(run_name="XGBoost_Final"):
    pipeline.fit(X_train, y_train)
    train_pred = pipeline.predict(X_train);
    val_pred = pipeline.predict(X_val)
    
    train_wmae = weighted_mae(y_train.values, train_pred, X_train["IsHoliday"].values)
    val_wmae   = weighted_mae(y_val.values,   val_pred,   X_val["IsHoliday"].values)
    val_mae    = float(np.mean(np.abs(y_val.values - val_pred)))
    val_rmse   = float(np.sqrt(np.mean((y_val.values - val_pred) ** 2)))
    overfit_gap = val_wmae - train_wmae
    
    mlflow.log_param("features", "all_features");
    mlflow.log_params(model.get_params())
    mlflow.log_metric("train_wmae", train_wmae);
    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("overfit_gap", overfit_gap);
    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_rmse", val_rmse)

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model",
        registered_model_name="XGBoost_Walmart",
        serialization_format="cloudpickle",
    )
    print("registered XGBoost_Walmart")
    
    print(f"train WMAE : {train_wmae:.2f}")
    print(f"val   WMAE : {val_wmae:.2f}")
    print(f"overfit gap: {overfit_gap:.2f}")
    print(f"val MAE    : {val_mae:.2f}")
    print(f"val RMSE   : {val_rmse:.2f}")

2026/07/08 16:12:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/08 16:12:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_Walmart'.
2026/07/08 16:12:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Walmart, version 1
Created version '1' of model 'XGBoost_Walmart'.


registered XGBoost_Walmart
train WMAE : 1329.75
val   WMAE : 1409.00
overfit gap: 79.25
val MAE    : 1409.20
val RMSE   : 2862.09
🏃 View run XGBoost_Final at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/0/runs/42783013286644dfadc179eab9f2c039
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/0
